In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy import stats
from tqdm import tqdm

from typing import List

import mne
from mne.io import read_raw_eeglab
from mne_connectivity import spectral_connectivity_epochs
from mne import make_forward_solution, setup_source_space, setup_volume_source_space
from mne.datasets import sample
from mne.io import read_raw_fif
from mne.minimum_norm import apply_inverse_epochs, make_inverse_operator
from mne.viz import circular_layout
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle
from mne.stats import fdr_correction

import random
import pickle
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


In [ ]:
import warnings


def load_eeg_data(
    file_path, eog: tuple = (), verbose: bool = True, picks: List[str] = None
):
    raw = read_raw_eeglab(
        input_fname=file_path,
        eog=eog,
        preload=True,
        montage_units="mm",
        verbose=verbose,
    )

    print(f"{raw.get_data().shape = }")
    raw = raw.pick(picks)
    print(f"{raw.get_data().shape = }")
    print(f"{raw.ch_names = }")
    raw.rename_channels(
        mapping={
            "FPZ": "Fpz",
            "OZ": "Oz",
            "FP1": "Fp1",
            "FP2": "Fp2",
            "FZ": "Fz",
            "FCZ": "FCz",
            "CPZ": "CPz",
            "CZ": "Cz",
            "PZ": "Pz",
            "POZ": "POz",
        }
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    raw.set_montage(montage)

    raw = raw.crop(tmin=10, tmax=raw.times[-1] - 10)
    raw = raw.resample(500, npad="auto")

    raw = raw.filter(l_freq=1, h_freq=50)
    raw = raw.interpolate_bads()
    # raw = raw.set_eeg_reference("average", projection=True)

    return raw


def load_metadata(filepath, condition_number=1):
    """Load and preprocess metadata."""
    metadata = pd.read_csv(filepath)
    metadata["channels"] = metadata["channels"].apply(eval)
    metadata = metadata[
        (metadata["condition_number"] == condition_number)
        & (~metadata["filename"].str.contains("COG"))
    ]
    return metadata


def preprocess_raw_data(raw: mne.io.Raw, smallers_duration: float):
    if raw.times[-1] <= smallers_duration:
        return raw

    total_duration = raw.times[-1]  # in seconds
    middle_duration = int(total_duration / 2)
    start = middle_duration - int(smallers_duration / 2)
    end = middle_duration + int(smallers_duration / 2)

    # print(f"{start = }")
    # print(f"{end = }")

    raw = raw.crop(tmin=start, tmax=end)

    return raw


def create_epochs_from_raw(
    raw, epoch_duration=2.0, overlap=0.0, tmin=0, tmax=None, baseline=None
):
    """
    Create epochs from raw MNE data.

    Parameters:
    -----------
    raw : mne.io.Raw
        The raw MNE data.
    epoch_duration : float, optional
        Duration of each epoch in seconds. Default is 2.0.
    overlap : float, optional
        Overlap between epochs in seconds. Default is 0.0 (no overlap).
    tmin : float, optional
        Start time of the epoch in seconds. Default is 0.
    tmax : float, optional
        End time of the epoch in seconds. If None, it will be set to epoch_duration.
    baseline : tuple or None, optional
        The baseline to apply. If None, no baseline is applied.

    Returns:
    --------
    epochs : mne.Epochs
        The created epochs.
    """
    # Create fixed-length events
    events = mne.make_fixed_length_events(raw, duration=epoch_duration, overlap=overlap)

    # If tmax is not specified, set it to epoch_duration
    if tmax is None:
        tmax = epoch_duration

    # Create epochs
    epochs = mne.Epochs(
        raw, events, tmin=tmin, tmax=tmax, baseline=baseline, preload=True
    )

    return epochs


def calculate_connectivity(
    epochs,
    method="wpli",
    fmin=8,
    fmax=13,
    faverage=True,
    mode="multitaper",
    mt_adaptive=False,
    n_jobs=1,
):
    """
    Calculate connectivity using spectral_connectivity_epochs.

    Parameters:
    -----------
    epochs : mne.Epochs
        The epochs to use for connectivity calculation.
    method : str, optional
        Connectivity method to use. Default is 'wpli'.
    fmin : float, optional
        Minimum frequency of interest. Default is 8 (lower bound of alpha band).
    fmax : float, optional
        Maximum frequency of interest. Default is 13 (upper bound of alpha band).
    faverage : bool, optional
        Whether to average across frequencies. Default is True.
    mode : str, optional
        Spectrum estimation mode. Default is 'multitaper'.
    mt_adaptive : bool, optional
        Whether to use adaptive weights for multitaper method. Default is False.
    n_jobs : int, optional
        Number of jobs to run in parallel. Default is 1.

    Returns:
    --------
    con : ndarray
        Connectivity matrix.
    freqs : ndarray
        Frequencies used in the calculation.
    times : ndarray
        Time points used in the calculation.
    n_epochs : int
        Number of epochs used.
    n_tapers : int
        Number of tapers used.
    """
    from mne_connectivity import spectral_connectivity_epochs

    sfreq = epochs.info["sfreq"]

    con = spectral_connectivity_epochs(
        epochs,
        method=method,
        mode=mode,
        sfreq=sfreq,
        fmin=fmin,
        fmax=fmax,
        faverage=faverage,
        tmin=epochs.tmin,
        mt_adaptive=mt_adaptive,
        n_jobs=n_jobs,
    )

    return con


def plot_connectivity(connectivity, method: str = "wpli"):
    """
    Plot connectivity matrix.

    Parameters:
    -----------
    connectivity : ndarray
        The connectivity matrix to plot. Should be a 3D array where the first two dimensions
        represent channels and the third dimension is 1 (as returned by spectral_connectivity_epochs).
    method : str, optional
        The connectivity method used, for the plot title. Default is "wpli".

    Returns:
    --------
    None. Displays the plot.
    """
    if not isinstance(connectivity, np.ndarray):
        connectivity = connectivity.get_data("dense")[:, :, 0]
    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(
        connectivity,
        vmin=0,
        vmax=1,
        cmap="tab20c",
        interpolation="nearest",
    )

    ax.set_title(f"{method.upper()} Connectivity")
    ax.set_ylabel("Channels")
    ax.set_xlabel("Channels")

    # Add colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(method.upper())

    plt.tight_layout()

    return fig


def detect_channel_sides(channels):
    """
    Detect whether each channel is on the left or right side.

    Args:
    channels (list): List of channel names.

    Returns:
    dict: Dictionary with channel names as keys and 'Left' or 'Right' as values.
    """

    def get_side(channel):
        numeric_part = "".join(filter(str.isdigit, channel))
        if numeric_part and int(numeric_part) % 2 == 0:
            return "Right"
        return "Left"

    return {channel: get_side(channel) for channel in channels}


def assign_channel_colors(channels_side, left_color, right_color):
    """
    Assign colors to channels based on their side.

    Args:
    channels_side (dict): Dictionary of channels and their sides.
    left_color (tuple): RGBA color for left channels.
    right_color (tuple): RGBA color for right channels.

    Returns:
    dict: Dictionary with channel names as keys and color tuples as values.
    """
    return {
        channel: left_color if side == "Left" else right_color
        for channel, side in channels_side.items()
    }


def assign_channel_5_colors():
    red = (0.458, 0.0, 0.0, 1.0)
    green = (0.0, 1.0, 0.0, 1.0)
    blue = (0.0, 0.0, 0.999, 1.0)
    purple = (0.5, 0.0, 0.5, 1.0)
    orange = (1.0, 0.647, 0.0, 1.0)

    channel_color = {
        # Frontal (Red)
        "AF3": red,
        "AF4": red,
        "F1": red,
        "F2": red,
        "F3": red,
        "F4": red,
        "F5": red,
        "F6": red,
        "F7": red,
        "F8": red,
        "FC1": red,
        "FC2": red,
        "FC3": red,
        "FC4": red,
        "FC5": red,
        "FC6": red,
        "FCZ": red,
        "FP1": red,
        "FP2": red,
        "FPZ": red,
        "FT7": red,
        "FT8": red,
        "FZ": red,
        # Central (Green)
        "C1": green,
        "C2": green,
        "C3": green,
        "C4": green,
        "C5": green,
        "C6": green,
        "CZ": green,
        # Parietal (Blue)
        "CP1": blue,
        "CP2": blue,
        "CP3": blue,
        "CP4": blue,
        "CP5": blue,
        "CP6": blue,
        "CPZ": blue,
        "P1": blue,
        "P2": blue,
        "P3": blue,
        "P4": blue,
        "P5": blue,
        "P6": blue,
        "P7": blue,
        "P8": blue,
        "PZ": blue,
        # Occipital (Purple)
        "O1": purple,
        "O2": purple,
        "OZ": purple,
        "PO3": purple,
        "PO4": purple,
        "PO5": purple,
        "PO6": purple,
        "PO7": purple,
        "PO8": purple,
        "POZ": purple,
        # Temporal (Orange)
        "T7": orange,
        "T8": orange,
        "TP7": orange,
        "TP8": orange,
    }

    return channel_color


def get_channel_5_colors():
    return [
        "AF3",
        "F1",
        "F3",
        "F5",
        "F7",
        "FC1",
        "FC3",
        "FC5",
        "FCZ",
        "FP1",
        "FPZ",
        "FT7",
        "FZ",
        "C1",
        "C3",
        "C5",
        "CZ",
        "T7",
        "CP1",
        "CP3",
        "CP5",
        "CPZ",
        "P1",
        "P3",
        "P5",
        "P7",
        "PZ",
        "PO3",
        "PO5",
        "PO7",
        "POZ",
        "O1",
        "OZ",
        "TP7",
        "TP8",
        "T8",
        "PO8",
        "PO6",
        "PO4",
        "O2",
        "P8",
        "P6",
        "P4",
        "P2",
        "CP6",
        "CP4",
        "CP2",
        "C6",
        "C4",
        "C2",
        "FT8",
        "FP2",
        "FC6",
        "FC4",
        "FC2",
        "F8",
        "F6",
        "F4",
        "F2",
        "AF4",
    ]


def prepare_circular_layout(label_names):
    """
    Prepare the circular layout for the connectivity plot.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles and node order for the circular layout.
    """
    n_channels = len(label_names)
    node_angles = circular_layout(label_names, label_names, start_pos=90)
    node_order = label_names
    return node_angles, node_order


def get_circular_plot_requirements(label_names):
    """
    Get the requirements for a circular connectivity plot with left channels first.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles, node order, and node colors for the circular plot.
    """
    # Define colors
    left_color = (0.0, 0.0, 0.999, 1.0)
    right_color = (0.458, 0.0, 0.0, 1.0)

    # Detect channel sides
    channels_side = detect_channel_sides(label_names)

    # Sort channels: left first, then right
    left_channels = [ch for ch, side in channels_side.items() if side == "Left"]
    right_channels = [ch for ch, side in channels_side.items() if side == "Right"]
    # sorted_channels = left_channels + right_channels

    sorted_channels = get_channel_5_colors()

    # Assign colors to channels
    # channel_color = assign_channel_colors(channels_side, left_color, right_color)
    channel_color = assign_channel_5_colors()

    # Prepare circular layout with sorted channels
    node_angles, _ = prepare_circular_layout(list(channel_color.keys()))

    # Get node colors in the same order as sorted_channels
    node_colors = [channel_color[channel] for channel in sorted_channels]

    return node_angles, sorted_channels, node_colors


def plot_circular_connectivity(
    connectivity,
    stage,
    label_names,
    node_angles,
    node_colors,
    method="wpli",
    is_save=False,
    n_lines=20,
    figsize=(8, 8),
):
    """
    Plot circular connectivity diagram.

    Args:
    connectivity (mne.Connectivity): The connectivity object.
    stage (str): The stage of the experiment (e.g., 'pre', 'during', 'post').
    label_names (list): List of channel names.
    node_angles (dict): Dictionary of node angles for the circular plot.
    node_colors (list): List of colors for each node.
    method (str, optional): Connectivity method used. Defaults to "wpli".
    is_save (bool, optional): Whether to save the figure. Defaults to False.
    n_lines (int, optional): Number of connections to draw. Defaults to 20.
    figsize (tuple, optional): Figure size. Defaults to (8, 8).

    Returns:
    matplotlib.figure.Figure: The created figure.
    """
    # Create figure
    fig, ax = plt.subplots(
        figsize=figsize, facecolor="black", subplot_kw=dict(polar=True)
    )

    # Plot connectivity circle
    plot_connectivity_circle(
        con=connectivity.get_data("dense")[:, :, 0],
        node_names=label_names,
        n_lines=n_lines,
        node_angles=node_angles,
        node_colors=node_colors,
        title=f"All-to-All Connectivity {stage} stimuli Condition ({method})",
        ax=ax,
    )

    # Adjust layout
    fig.tight_layout()

    # Save figure if requested
    if is_save:
        filename = f"{method}_{stage}.png"
        fig.savefig(filename, dpi=300)
        print(f"Figure saved as {filename}")

    return fig


def save_connectivity(con, filepath):
    """Save connectivity data."""
    with open(filepath, "wb") as f:
        pickle.dump(con, f)


def process_patient_data(metadata, patient, stage_no, smallest_duration, output_dir):
    """Process data for a single patient and stage."""
    patient_data = metadata[
        (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
    ]
    if patient_data.empty:
        logging.warning(f"No data found for patient {patient}, stage {stage_no}")
        return

    patient_data = patient_data.iloc[0]
    filepath, channels = patient_data[["filepath", "channels"]]

    raw = load_eeg_data(filepath, picks=channels)
    raw = preprocess_raw_data(raw, smallest_duration)
    epochs = create_epochs_from_raw(raw)
    con = calculate_connectivity(epochs)

    # Save connectivity data
    if stage_no == 1:
        stage_name = "pre"
    elif stage_no == 2:
        stage_name = "during"
    else:
        stage_name = "post"

    save_filepath = os.path.join(output_dir, f"{patient}_{stage_name}_connectivity.pkl")
    save_connectivity(con, save_filepath)

    # Plot and save connectivity
    plt.figure(figsize=(10, 8))
    fig = plot_connectivity(con, method="wpli")
    plt.savefig(
        os.path.join(output_dir, f"{patient}_{stage_name}_connectivity_plot.png")
    )
    plt.close()

    # Plot and save circular connectivity
    node_angles, node_order, node_colors = get_circular_plot_requirements(channels)
    fig = plot_circular_connectivity(
        con, stage_name, node_order, node_angles, node_colors
    )
    fig.savefig(os.path.join(output_dir, f"{patient}_{stage_name}_circular_plot.png"))
    plt.close(fig)


def get_patient_epoch_data(metadata, patient, stage_no, smallest_duration):
    """Process data for a single patient and stage."""
    patient_data = metadata[
        (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
    ]
    if patient_data.empty:
        logging.warning(f"No data found for patient {patient}, stage {stage_no}")
        return

    patient_data = patient_data.iloc[0]
    filepath, channels = patient_data[["filepath", "channels"]]

    raw = load_eeg_data(filepath, picks=channels)
    raw = preprocess_raw_data(raw, smallest_duration)
    epochs = create_epochs_from_raw(raw)
    return epochs


def get_patient_conn_data(metadata, patient, stage_no, smallest_duration):
    """Process data for a single patient and stage."""
    patient_data = metadata[
        (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
    ]
    if patient_data.empty:
        logging.warning(f"No data found for patient {patient}, stage {stage_no}")
        return

    patient_data = patient_data.iloc[0]
    filepath, channels = patient_data[["filepath", "channels"]]

    raw = load_eeg_data(filepath, picks=channels)
    raw = preprocess_raw_data(raw, smallest_duration)
    epochs = create_epochs_from_raw(raw)
    con = calculate_connectivity(epochs, mt_adaptive=True)

    return con.get_data("dense")


def perform_ttest_and_fdr_correction(
    pre_con, post_con, n_chan: int = 60, output_dir: str = ""
):
    def safe_ttest_rel(a, b):
        """Perform a paired t-test with handling for zero-variance cases."""
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # Suppress warnings
            try:
                t, p = stats.ttest_rel(a, b)
                print(f"{t = }, {p = }")
                if np.isnan(t) or np.isnan(p):
                    return 0, 1  # No significant difference
                return t, p
            except:
                return 0, 1  # Return no significant difference in case of error

    if isinstance(pre_con, list):
        for i in range(len(pre_con)):
            print(f"{i}: {pre_con[i].shape = }")
        pre_con = np.array(pre_con)
    if isinstance(post_con, list):
        post_con = np.array(post_con)

    print(f"{pre_con.shape = }")
    print(f"{post_con.shape = }")

    # log mean and std value of pre and post
    logging.info(f"Pre mean: {np.mean(pre_con)}")
    logging.info(f"Pre std: {np.std(pre_con)}")
    logging.info(f"Post mean: {np.mean(post_con)}")
    logging.info(f"Post std: {np.std(post_con)}")

    t_scores = np.zeros((n_chan, n_chan))
    p_values = np.zeros((n_chan, n_chan))

    for i in range(n_chan):
        for j in range(i + 1, n_chan):
            pre_stim = np.array([wpli[j, i, 0] for wpli in pre_con])
            post_stim = np.array([wpli[j, i, 0] for wpli in post_con])

            print(f"{pre_stim.shape = }")
            print(f"{post_stim.shape = }")

            # Check if there's any variance in the data
            if len(pre_stim) > 1 and len(post_stim) > 1:
                if np.var(pre_stim) == 0 and np.var(post_stim) == 0:
                    t_scores[j, i], p_values[j, i] = 0, 1  # No difference
                else:
                    t_scores[j, i], p_values[j, i] = safe_ttest_rel(pre_stim, post_stim)

            else:
                t_scores[j, i], p_values[j, i] = safe_ttest_rel(pre_stim, post_stim)

    # Make the matrices symmetric
    t_scores = t_scores + t_scores.T
    p_values = p_values + p_values.T

    print(f"{t_scores = }")
    print(f"{p_values = }")

    # Step 5: Apply FDR correction
    reject, fdr_pvals = fdr_correction(p_values.flatten())
    fdr_pvals = fdr_pvals.reshape(p_values.shape)

    logging.info(f"P value: {p_values.shape}")
    logging.info(f"fdr value: {fdr_pvals.shape}")

    # Step 6: Create a mask of significant results
    sig_mask = fdr_pvals < 0.05
    print(f"{fdr_pvals = }")
    print(f"{sig_mask = }")

    # Save the figure instead of showing itplt.figure(figsize=(10, 8))

    if output_dir != "":
        plt.imshow(sig_mask, cmap="binary", interpolation="nearest")
        plt.colorbar(label="Significant Connection")
        plt.title(
            "Significant WPLI Changes (Pre- vs Post-Stimulus, FDR-corrected p < 0.05)"
        )
        plt.xlabel("Channels")
        plt.ylabel("Channels")
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, "wpli_significant_changes.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()  # Close the figure to free up memory

        # Step 8: Save the results
        np.save(os.path.join(output_dir, "fdr_corrected_pvalues.npy"), fdr_pvals)
        np.save(os.path.join(output_dir, "significant_mask.npy"), sig_mask)

    return p_values, fdr_pvals, sig_mask


def plot_condition_state(exp_output, condition, state1, state2):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Condition {condition}, States {state1} vs {state2}")

    key = (state1, state2)
    data = exp_output[condition][key]

    titles = ["p-values", "FDR-corrected p-values", "Significance Mask"]
    values = ["p_values", "fdr_pvals", "sig_mask"]
    cmaps = ["viridis", "viridis", "binary"]

    for i, (v, title, cmap) in enumerate(zip(values, titles, cmaps)):
        im = axs[i].imshow(data[v], cmap=cmap)
        axs[i].set_title(title)
        plt.colorbar(im, ax=axs[i])

    plt.tight_layout()
    plt.show()


def plot_all_condition_comparisons(
    exp2_output,
    state,
    figsize=(25, 5 * 15),
    output_dir: str = "",
    filename_prefix: str = "during_among_conditions",
    dpi: int = 300,
):
    """
    Plot heatmaps for all condition comparisons and all value types for a given state.

    Parameters:
    exp2_output (dict): The output dictionary containing all comparisons
    state (str): The state to plot ('during' or 'post')
    figsize (tuple): The size of the overall figure
    """

    comparisons = list(exp2_output.keys())
    n_comparisons = len(comparisons)
    value_types = ["p_values", "fdr_pvals", "sig_mask"]

    fig, axes = plt.subplots(n_comparisons, 3, figsize=figsize)
    fig.suptitle(f"{state.capitalize()} State - All Comparisons", fontsize=16)

    # Initialize min and max values for consistent color scaling
    vmin, vmax = {}, {}
    for vtype in value_types:
        vmin[vtype], vmax[vtype] = float("inf"), float("-inf")

    # First pass to determine global min and max for each value type
    for comparison in comparisons:
        for vtype in value_types:
            data = exp2_output[comparison][state][vtype]
            vmin[vtype] = min(vmin[vtype], np.min(data))
            vmax[vtype] = max(vmax[vtype], np.max(data))

    for i, comparison in enumerate(comparisons):
        for j, vtype in enumerate(value_types):
            data = exp2_output[comparison][state][vtype]

            # Use a different colormap for sig_mask
            cmap = "binary" if vtype == "sig_mask" else "viridis"

            im = axes[i, j].imshow(data, cmap=cmap, vmin=vmin[vtype], vmax=vmax[vtype])
            axes[i, j].set_title(f"{comparison} - {vtype}")
            axes[i, j].axis("off")

            # Add colorbar for each subplot
            plt.colorbar(im, ax=axes[i, j], fraction=0.046, pad=0.04)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to prevent overlap

    if output_dir != "":
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Save the figure
        filename = f"{filename_prefix}_{state}.png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=dpi, bbox_inches="tight")

        print(f"Image saved as {filepath}")

        plt.close(fig)  # Close the figure to free up memory

def save_experiment_results(exp_output, output_folder):
    """
    Save experiment results in a specified folder.

    Parameters:
    -----------
    exp_output : dict
        Dictionary containing the experiment output.
    output_folder : str
        Path to the output folder.

    Returns:
    --------
    None
    """
    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Iterate over the experiment output
    for experiment_name, conditions in exp_output.items():
        experiment_folder = os.path.join(output_folder, str(experiment_name))
        os.makedirs(experiment_folder, exist_ok=True)

        # Iterate over the conditions
        for condition, results in conditions.items():
            condition_folder = os.path.join(
                experiment_folder, f"Condition_{str(condition)}"
            )
            os.makedirs(condition_folder, exist_ok=True)

            # Save the results
            stage_folder = os.path.join(condition_folder)
            os.makedirs(stage_folder, exist_ok=True)

            # Save p_values
            p_values_file = os.path.join(stage_folder, "p_values.npy")
            np.save(p_values_file, results["p_values"])

            # Save fdr_pvals
            fdr_pvals_file = os.path.join(stage_folder, "fdr_pvals.npy")
            np.save(fdr_pvals_file, results["fdr_pvals"])

            # Save sig_mask
            sig_mask_file = os.path.join(stage_folder, "sig_mask.npy")
            np.save(sig_mask_file, results["sig_mask"])

    print("Experiment results saved successfully.")


In [ ]:
def main():
    metadata_path = "../metadata.csv"
    conditions = 1
    output = {}
    for condition in range(1, conditions + 1):
        metadata = load_metadata(metadata_path, condition)
        smallest_duration = metadata["duration"].min()
        logging.info(f"Smallest duration: {smallest_duration}")

        connectivities = {1: [], 2: [], 3: []}
        for patient in range(2, 22):  # Process patients 2 to 21
            if patient == 16:
                continue  # Skip patient 16
            for stage_no in connectivities.keys():  # pre and post
                try:
                    con = get_patient_conn_data(
                        metadata, patient, stage_no, smallest_duration
                    )

                    connectivities[stage_no].append(con)
                    logging.info(
                        f"Processed data for patient {patient}, stage {stage_no}"
                    )
                except Exception as e:
                    logging.error(
                        f"Error processing data for patient {patient}, stage {stage_no}: {e}"
                    )

        output[condition] = connectivities

    return output


connetivities = main()


In [ ]:
temp = connetivities.copy()


In [ ]:
connetivities.keys()


In [ ]:
patients = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21]

for condition, connectivity in connetivities.items():
    for stage_no, con in connectivity.items():
        if len(con) > 0:
            # only print some values
            print(f"{condition = }, {stage_no = }, {len(con) = }")

            for i, con_data in zip(patients, con):
                if con_data is not None:
                    print(f"\t{i}: {con_data.shape = }")
                else:
                    print(f"\t{i}: {con_data = }")
        else:
            print(f"{condition = }, {stage_no = }, No data found")


In [ ]:
# Experimnet 1: Pre vs Post and Pre vs During


def sanity_check(con):
    # return only value of con that is not None
    # iterate over value of con and check if it is not None
    # append them to a list and then return the list as numpy array
    return np.array([c for c in con if c is not None])


patients = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21]
states = [(1, 3), (1, 2)]  # 1: pre, 3: post, 2: during

exp1_output = {}
for condition, connectivity in tqdm(connetivities.items()):

    states_output = {}
    for s in states:
        con_1 = connectivity[s[0]]
        con_2 = connectivity[s[1]]
        con_1 = sanity_check(con_1)
        con_2 = sanity_check(con_2)

        patient_output = {}
        try:

            p_values, fdr_pvals, sig_mask = perform_ttest_and_fdr_correction(
                con_1, con_2, n_chan=60, output_dir=""
            )

            states_output[s] = {
                "p_values": p_values,
                "fdr_pvals": fdr_pvals,
                "sig_mask": sig_mask,
            }

        except Exception as e:
            states_output[s] = {
                "p_values": np.zeros((60, 60)),
                "fdr_pvals": np.zeros((60, 60)),
                "sig_mask": np.zeros((60, 60)),
            }
    exp1_output[condition] = states_output


In [ ]:

fdr_results_path = (
    "D:\\neuro-closeloop-project\\vielight_close_loop\\notebooks\\results_fdr"
)
output_folder = os.path.join(fdr_results_path, "experiment_1")
save_experiment_results(exp1_output, output_folder)


In [ ]:
print(exp1_output.keys())
print(exp1_output[1].keys())
print(exp1_output[1][(1, 3)].keys())
print(exp1_output[1][(1, 3)]["p_values"].shape)


In [ ]:
for condition in range(1, 7):
    plot_condition_state(exp1_output, condition, 1, 3)
    plot_condition_state(exp1_output, condition, 1, 2)


In [ ]:
connectivities = temp.copy()


In [ ]:
connectivities.keys()


In [ ]:
# Experimnet 2: Post
from itertools import combinations


def sanity_check(con):
    # return only value of con that is not None
    # iterate over value of con and check if it is not None
    # append them to a list and then return the list as numpy array
    return np.array([c for c in con if c is not None])


patients = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21]
states = {"during": 2, "post": 3}
conditions = list(connectivities.keys())
condition_pairs = list(combinations(conditions, 2))

exp2_output = {}

for condition_pair in tqdm(condition_pairs):
    cond1, cond2 = condition_pair
    states_output = {}

    for state_name, state_value in states.items():

        con_1 = sanity_check(connectivities[cond1][state_value])
        con_2 = sanity_check(connectivities[cond2][state_value])

        print(f"{cond1 = }, {cond2 = }, {state_name = }, {state_value = }")
        print(f"{con_1.shape = }, {con_2.shape = }")

        try:
            p_values, fdr_pvals, sig_mask = perform_ttest_and_fdr_correction(
                con_1, con_2, n_chan=60, output_dir=""
            )
            states_output[state_name] = {
                "p_values": p_values,
                "fdr_pvals": fdr_pvals,
                "sig_mask": sig_mask,
            }

        except Exception as e:
            states_output[state_name] = {
                "p_values": np.zeros((60, 60)),
                "fdr_pvals": np.zeros((60, 60)),
                "sig_mask": np.zeros((60, 60)),
            }
            print(f"Error processing {cond1} vs {cond2} for {state_name}: {str(e)}")

    exp2_output[f"{cond1}_vs_{cond2}"] = states_output


In [ ]:
print(exp2_output.keys())
print(exp2_output["1_vs_2"].keys())
print(exp2_output["1_vs_2"]["during"].keys())
print(exp2_output["1_vs_2"]["during"]["p_values"].shape)


In [ ]:
plot_all_condition_comparisons(
    exp2_output, "during", output_dir=".", filename_prefix="during_among_conditions"
)
plot_all_condition_comparisons(
    exp2_output, "post", output_dir=".", filename_prefix="post_among_conditions"
)


In [ ]:
def main():
    metadata_path = "../metadata.csv"
    condition_no = 5
    metadata = load_metadata(metadata_path, condition_no)
    smallest_duration = metadata["duration"].min()
    logging.info(f"Smallest duration: {smallest_duration}")

    connectivities = {1: [], 3: []}
    for patient in range(2, 22):  # Process patients 2 to 21
        if patient == 16:
            continue  # Skip patient 16
        for stage_no in connectivities.keys():  # pre and post
            try:
                con = get_patient_conn_data(
                    metadata, patient, stage_no, smallest_duration
                )

                connectivities[stage_no].append(con)
                logging.info(f"Processed data for patient {patient}, stage {stage_no}")
            except Exception as e:
                logging.error(
                    f"Error processing data for patient {patient}, stage {stage_no}: {e}"
                )

    # Perform t-test and FDR correction
    output_dir = f"fdr/{condition_no}/results_fdr"
    os.makedirs(output_dir, exist_ok=True)
    perform_ttest_and_fdr_correction(
        connectivities[1], connectivities[3], output_dir=output_dir
    )  # 1 is pre, 3 is post


if __name__ == "__main__":
    main()


In [ ]:
results_path = (
    "D:\\neuro-closeloop-project\\vielight_close_loop\\notebooks\\results_fdr"
)

temp = np.load(os.path.join(results_path, "fdr_corrected_pvalues.npy"))


This script demonstrates three different scenarios in WPLI analysis:

Scenario 1: No significant difference

Both pre- and post-stimulus data are generated with the same mean and standard deviation.
We expect to see very few, if any, significant differences.


Scenario 2: Significant difference

Pre-stimulus data has a lower mean than post-stimulus data.
We expect to see many significant differences.


Scenario 3: Zero variance in some channels

Similar to Scenario 1, but with zero variance introduced in one channel pair.
This tests our handling of zero-variance cases.



Key components of the script:

Synthetic Data Generation:

We use numpy.random.normal to generate synthetic WPLI data.
Different means and standard deviations are used to create various scenarios.


Analysis Function:

safe_ttest_rel performs a paired t-test while handling potential errors.
analyze_wpli applies the t-test to all channel pairs and performs FDR correction.


Visualization:

We create a 3x2 grid of plots, showing t-scores and significant connections for each scenario.
T-scores are plotted using a diverging colormap to show both positive and negative differences.
Significant connections (after FDR correction) are plotted as a binary mask.



Interpretation of results:

Scenario 1: You should see mostly non-significant results (few or no black dots in the significance plot).
Scenario 2: You should see many significant results (many black dots in the significance plot).
Scenario 3: Similar to Scenario 1, but with one channel pair (0,1) showing zero variance (likely appearing as a distinct pattern in the t-score plot).

This synthetic data analysis helps in understanding how the WPLI analysis behaves under different conditions and verifies that our statistical approach, including FDR correction and handling of zero-variance cases, works as expected.
To use this with your real data, you would replace the synthetic data generation with your actual WPLI computations from EEG data, keeping the analysis and visualization parts largely the same.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from mne.stats import fdr_correction
import warnings


# Synthetic data generation
def generate_synthetic_wpli(n_patients, n_channels, mean, std):
    return np.random.normal(mean, std, (n_patients, n_channels, n_channels, 1))


# Parameters
n_patients = 21
n_channels = 10

# Generate synthetic WPLI data for different scenarios
np.random.seed(42)  # for reproducibility

# Scenario 1: No significant difference
wpli_pre_1 = generate_synthetic_wpli(n_patients, n_channels, 0.5, 0.1)
wpli_post_1 = generate_synthetic_wpli(n_patients, n_channels, 0.5, 0.1)

# Scenario 2: Significant difference
wpli_pre_2 = generate_synthetic_wpli(n_patients, n_channels, 0.1, 0.1)
wpli_post_2 = generate_synthetic_wpli(n_patients, n_channels, 0.9, 0.1)

# Scenario 3: Zero variance in some channels
wpli_pre_3 = generate_synthetic_wpli(n_patients, n_channels, 0.5, 0.1)
wpli_post_3 = generate_synthetic_wpli(n_patients, n_channels, 0.5, 0.1)
wpli_pre_3[:, 0, 1] = 0.5  # Zero variance for channel pair (0,1)
wpli_post_3[:, 0, 1] = 0.5


# Analysis function
def safe_ttest_rel(a, b):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            t, p = stats.ttest_rel(a, b)
            if np.isnan(t) or np.isnan(p):
                return 0, 1
            return t, p
        except:
            return 0, 1


def analyze_wpli(wpli_pre, wpli_post):
    n_chan = wpli_pre.shape[1]
    t_scores = np.zeros((n_chan, n_chan))
    p_values = np.zeros((n_chan, n_chan))

    for i in range(n_chan):
        for j in range(i + 1, n_chan):
            pre_stim = wpli_pre[:, i, j, 0]
            post_stim = wpli_post[:, i, j, 0]

            if np.var(pre_stim) == 0 and np.var(post_stim) == 0:
                t_scores[i, j], p_values[i, j] = 0, 1
            else:
                t_scores[i, j], p_values[i, j] = safe_ttest_rel(pre_stim, post_stim)

    # Make matrices symmetric
    t_scores = t_scores + t_scores.T
    p_values = p_values + p_values.T

    # FDR correction
    _, fdr_pvals = fdr_correction(p_values.flatten())
    fdr_pvals = fdr_pvals.reshape(p_values.shape)

    return t_scores, fdr_pvals


# Analyze each scenario
results = []
for scenario, (wpli_pre, wpli_post) in enumerate(
    [(wpli_pre_1, wpli_post_1), (wpli_pre_2, wpli_post_2), (wpli_pre_3, wpli_post_3)], 1
):
    t_scores, fdr_pvals = analyze_wpli(wpli_pre, wpli_post)

    print(f"{scenario = }")
    print(f"{t_scores = }")
    print(f"\n{fdr_pvals =}")

    results.append((t_scores, fdr_pvals))

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(15, 20))
for i, (t_scores, fdr_pvals) in enumerate(results):
    ax_t = axes[i, 0]
    ax_p = axes[i, 1]

    im_t = ax_t.imshow(t_scores, cmap="coolwarm", vmin=-5, vmax=5)
    im_p = ax_p.imshow(fdr_pvals < 0.05, cmap="binary")

    ax_t.set_title(f"Scenario {i+1}: T-scores")
    ax_p.set_title(f"Scenario {i+1}: Significant Connections (FDR < 0.05)")

    fig.colorbar(im_t, ax=ax_t)
    fig.colorbar(im_p, ax=ax_p)

plt.tight_layout()
plt.savefig("wpli_analysis_scenarios.png", dpi=300, bbox_inches="tight")
plt.close()

print(
    "Analysis complete. Visualization has been saved as 'wpli_analysis_scenarios.png'."
)
